<a href="https://colab.research.google.com/github/solasobambo-prog/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/solasobambo-prog/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [40]:
from huggingface_hub import login
from google.colab import userdata

hf_token = userdata.get('HF_TOKEN')
login(token=hf_token)

print("Login successful, token loaded from Colab Secrets.")

Login successful, token loaded from Colab Secrets.


In [41]:
from huggingface_hub import HfApi

api = HfApi()
files = api.list_repo_files("FlyRank/internship-warehouse", repo_type="dataset")
print(f"Found {len(files)} files in the warehouse repo.")
print(files[:10])

Found 24 files in the warehouse repo.
['.gitattributes', 'README.md', 'dim_clients.parquet', 'dim_content.parquet', 'fact_content_daily_performance/month=2025-01/data_0.parquet', 'fact_content_daily_performance/month=2025-02/data_0.parquet', 'fact_content_daily_performance/month=2025-03/data_0.parquet', 'fact_content_daily_performance/month=2025-04/data_0.parquet', 'fact_content_daily_performance/month=2025-05/data_0.parquet', 'fact_content_daily_performance/month=2025-06/data_0.parquet']


## 1. Unit of analysis + time window

**One row = one content page, on one report date, for one client** (a page-day), in
the `fact_content_daily_performance` table. Grain columns: `report_date`,
`client_hash_id`, `content_hash_id`.

**Time window:** report_date within March 2026 (`month=2026-03`), a mid-panel month,
per the internship's iteration rule (never the `_sample` partition, which is June 2026,
the sealed final month).

In [42]:
import duckdb
import os
from huggingface_hub import hf_hub_download

# Download just the mid-panel month partition (2026-03), not the _sample or full warehouse
month_file = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    repo_type="dataset",
    filename="fact_content_daily_performance/month=2026-03/data_0.parquet",
    token=hf_token
)

con = duckdb.connect()
con.execute(f"CREATE VIEW fact_march AS SELECT * FROM read_parquet('{month_file}')")

print("Loaded month=2026-03 partition.")
print(con.execute("SELECT COUNT(*) AS row_count FROM fact_march").fetchdf())


Loaded month=2026-03 partition.
   row_count
0    9841378


## 2. Fields: feature / label / context / excluded

**Feature (knowable before the decision moment, safe to use):**
- `gsc_avg_position`, current ranking position, known at the moment of scoring, not
  derived from the outcome I am proxying.
- `gsc_impressions`, search visibility volume, needed to judge how much a page's CTR
  can be trusted (per the volume lesson from ML-01/ML-02).
- `ga4_engaged_sessions`, on-page engagement signal, available at the same moment, and
  distinct from click through behavior itself.
- `scroll_events`, another engagement signal, same reasoning as above.
- `sessions_organic`, organic traffic volume, a different lens on visibility than GSC
  impressions alone.

**Label / proxy (the thing I predict or rank, or what it is computed from, never a
feature):**
- `gsc_clicks` and `gsc_impressions` together compute `ctr` (clicks divided by
  impressions), and `ctr` compared against the tier average produces my `ctr_gap`
  proxy from ML-03. Note: `gsc_impressions` appears in both lists above, once as a
  feature (raw visibility/volume signal) and once inside the label's own formula. I
  will keep this in mind while building the model stage, using impressions as a
  trust/volume filter is different from using it to predict ctr_gap itself, but this
  needs care so it does not quietly become circular.

**Context (for grouping, joining, splitting, reading, never for the model to learn
from):**
- `report_date`, defines the time window, used for splitting/filtering, not as a
  learned signal.
- `client_hash_id`, pseudonymized client identifier, used only for grouped train/test
  splits later, never a feature.
- `content_hash_id`, pseudonymized page identifier, the grain key, used for joining
  and identifying rows, never a feature.
- `month`, partition label, used for selecting the right file, not a signal.

**Excluded (private, product-decision flags, or future information, each with a
why):**
- `client_has_gsc`, `client_has_ga4`: client-level configuration flags, not a per-page
  signal, and including them risks the model learning client identity patterns rather
  than page-level behavior.
- `ai_chatgpt`, `ai_perplexity`, `ai_gemini`, `ai_copilot`, `ai_claude`, `ai_meta`,
  `ai_other`: AI referral breakdown columns are out of scope for Lane 4 (CTR/engagement
  scoring is about search click behavior, not AI-assistant referral sources), and per
  the flyrank-data skill, sparse/uneven history makes this its own separate research
  question, not something to fold in here.
- `sessions_paid`, `sessions_direct`, `sessions_referral`, `sessions_social`: other
  traffic-source breakdowns, excluded for this week to keep the five-feature limit
  focused on signals directly tied to the CTR/position framing, not because they are
  unusable in general.
- `gsc_sum_position`: a raw sum rather than an average, redundant once `gsc_avg_position`
  is available, and easy to misuse if divided incorrectly.

In [43]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


# 3. Verify it with queries (grain, counts, missing values, windows)

In [44]:
# Section 3, Query 1: Verify grain: one row = one (report_date, client_hash_id, content_hash_id) combination
grain_check = con.execute("""
    SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS c
    FROM fact_march
    GROUP BY report_date, client_hash_id, content_hash_id
    HAVING COUNT(*) > 1
    LIMIT 5
""").fetchdf()

print(f"Rows violating the grain: {len(grain_check)}")
grain_check

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows violating the grain: 0


,report_date,client_hash_id,content_hash_id,c


**Query 2 (row count, date span, verifies Section 1's time window claim):**

Filtering to `gsc_data_available = TRUE` for March 2026 gives 3,611,061 rows (about
37% of the full month's 9,841,378 rows), spanning the complete month (2026-03-01 to
2026-03-31), across 176,738 unique pages and 47 unique clients. The large drop from
the full partition confirms the flyrank-data skill's warning that a meaningful share
of clients or page-days lack usable search data, this is not a bug, it just means my
slice is smaller than the raw table suggests.

In [45]:
# Query 2: row count and date span for the Lane 4 slice (GSC data, since CTR needs impressions/clicks)
lane4_slice = con.execute("""
    SELECT
        COUNT(*) AS row_count,
        MIN(report_date) AS min_date,
        MAX(report_date) AS max_date,
        COUNT(DISTINCT content_hash_id) AS unique_pages,
        COUNT(DISTINCT client_hash_id) AS unique_clients
    FROM fact_march
    WHERE gsc_data_available = TRUE
""").fetchdf()

print(lane4_slice.to_string())


   row_count   min_date   max_date  unique_pages  unique_clients
0    3611061 2026-03-01 2026-03-31        176738              47


In [46]:
# Query 3: availability check with IS TRUE, showing how many rows survive
availability_check = con.execute("""
    SELECT
        COUNT(*) AS total_rows,
        SUM(CASE WHEN gsc_data_available IS TRUE THEN 1 ELSE 0 END) AS gsc_available_rows,
        SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) AS ga4_available_rows,
        SUM(CASE WHEN gsc_data_available IS TRUE AND ga4_data_available IS TRUE THEN 1 ELSE 0 END) AS both_available_rows
    FROM fact_march
""").fetchdf()

print(availability_check.to_string())

   total_rows  gsc_available_rows  ga4_available_rows  both_available_rows
0     9841378           3611061.0            413966.0             364347.0


**Query 3 (availability check with IS TRUE, verifies field classification decisions in Section 2):**

Of the 9,841,378 total rows in March 2026, only 3,611,061 (36.7%) have
`gsc_data_available IS TRUE`, and just 413,966 (4.2%) have `ga4_data_available IS TRUE`.
Rows with both available drop to 364,347 (3.7%). This confirms that GSC data is far
more complete than GA4 in this warehouse, which shapes my feature choices: my Lane 4
proxy (ctr_gap) only needs GSC fields (gsc_clicks, gsc_impressions, gsc_avg_position),
so I can safely rely on the larger GSC-available slice. Any future feature pulling from
GA4 (engagement time, sessions) would need to account for a much smaller, likely biased
subset of pages.

In [47]:
# Missing values check: overall null rate for key feature columns
missing_check = con.execute("""
    SELECT
        AVG(CASE WHEN gsc_avg_position IS NULL THEN 1.0 ELSE 0 END) AS pct_null_avg_position,
        AVG(CASE WHEN gsc_impressions IS NULL THEN 1.0 ELSE 0 END) AS pct_null_impressions,
        AVG(CASE WHEN ga4_engaged_sessions IS NULL THEN 1.0 ELSE 0 END) AS pct_null_ga4_engaged,
        AVG(CASE WHEN scroll_events IS NULL THEN 1.0 ELSE 0 END) AS pct_null_scroll,
        AVG(CASE WHEN sessions_organic IS NULL THEN 1.0 ELSE 0 END) AS pct_null_sessions_organic
    FROM fact_march
    WHERE gsc_data_available IS TRUE
""").fetchdf()

print(missing_check.to_string())

   pct_null_avg_position  pct_null_impressions  pct_null_ga4_engaged  pct_null_scroll  pct_null_sessions_organic
0                    0.0                   0.0              0.423246         0.423246                   0.423246


In [48]:
# Does missingness follow a pattern? Check ga4_engaged_sessions nulls by client (top 10 by row count)
pattern_check = con.execute("""
    SELECT
        client_hash_id,
        COUNT(*) AS total_rows,
        AVG(CASE WHEN ga4_engaged_sessions IS NULL THEN 1.0 ELSE 0 END) AS pct_null_ga4
    FROM fact_march
    WHERE gsc_data_available IS TRUE
    GROUP BY client_hash_id
    ORDER BY total_rows DESC
    LIMIT 10
""").fetchdf()

print(pattern_check.to_string())

            client_hash_id  total_rows  pct_null_ga4
0  client_73cda7b4e4f265ea      725539      0.631605
1  client_62f4a7e64f5e0096      610971      1.000000
2  client_23a62021009f63c4      388204      0.000000
3  client_08a6a72ff48e62c0      359419      1.000000
4  client_e547b89c05043229      255933      0.000000
5  client_fef1a8f436438636      254118      0.146798
6  client_a80fca3f171ed1de      124269      0.056458
7  client_20259bd6705d81d4      119463      0.000000
8  client_e5c2aa26a8598242       82320      0.162451
9  client_3f0ce4d44fe94f3d       81166      0.046941


**Missing values check (verifies field classification and Section 4 limitations):**

Overall, `gsc_avg_position` and `gsc_impressions` have 0% nulls among GSC-available
rows, fully reliable. But `ga4_engaged_sessions`, `scroll_events`, and
`sessions_organic` are all null at exactly the same rate (42.3%), which itself is a
clue, these three likely come from the same underlying GA4 pull, missing together as
a bloc.

Checking by client confirms this is not random: some clients (e.g.
client_62f4a7e64f5e0096, client_08a6a72ff48e62c0) show 100% nulls, meaning they have
no GA4 connected at all, while others (e.g. client_23a62021009f63c4,
client_e547b89c05043229) show 0% nulls, full coverage. This matches the flyrank-data
skill's warning: GA4 missingness follows client-level availability, not random chance.

**Consequence for my feature set:** using ga4_engaged_sessions or scroll_events as a
feature means roughly 42% of rows would need a missing-value strategy, and a blind
fillna(0) would be wrong, it would make an unconnected client's pages look like they
have zero engagement, when the truth is "unmeasured," not "measured as zero." Per the
flyrank-data skill, the correct fix is a has_ga4 flag alongside the raw value, not a
silent zero-fill.

In [49]:
# Windows check: date span per client (top 10 by row count), to see if histories align
window_check = con.execute("""
    SELECT
        client_hash_id,
        MIN(report_date) AS min_date,
        MAX(report_date) AS max_date,
        COUNT(*) AS row_count
    FROM fact_march
    WHERE gsc_data_available IS TRUE
    GROUP BY client_hash_id
    ORDER BY row_count DESC
    LIMIT 10
""").fetchdf()

print(window_check.to_string())

            client_hash_id   min_date   max_date  row_count
0  client_73cda7b4e4f265ea 2026-03-01 2026-03-31     725539
1  client_62f4a7e64f5e0096 2026-03-01 2026-03-31     610971
2  client_23a62021009f63c4 2026-03-01 2026-03-31     388204
3  client_08a6a72ff48e62c0 2026-03-01 2026-03-31     359419
4  client_e547b89c05043229 2026-03-01 2026-03-31     255933
5  client_fef1a8f436438636 2026-03-01 2026-03-31     254118
6  client_a80fca3f171ed1de 2026-03-01 2026-03-31     124269
7  client_20259bd6705d81d4 2026-03-01 2026-03-31     119463
8  client_e5c2aa26a8598242 2026-03-01 2026-03-31      82320
9  client_3f0ce4d44fe94f3d 2026-03-01 2026-03-31      81166


**Windows check (verifies Section 1's time window claim, per client):**

Checking the top 10 clients by row count, every one has full coverage across the
entire March window (2026-03-01 to 2026-03-31), no partial-month starts or ends among
the largest clients. This is a limited check though, it only covers the top 10 of 47
total clients in this slice; smaller clients were not individually verified and could
have partial-month histories. Per the flyrank-data skill's warning that client
history depth varies widely, I would need to check dim_clients.gsc_data_start before
assuming every client has a full month, especially for any client outside this top 10.

In [50]:
# Section 3: Five-feature frame, a real cross-section using reservoir sampling
feature_frame = con.execute("""
    SELECT
        content_hash_id,
        client_hash_id,
        report_date,
        gsc_avg_position,
        gsc_impressions,
        ga4_engaged_sessions,
        scroll_events,
        sessions_organic,
        ga4_data_available AS has_ga4_data
    FROM fact_march
    WHERE gsc_data_available IS TRUE
    USING SAMPLE 20000 (reservoir)
""").fetchdf()

print(f"Feature frame shape: {feature_frame.shape}")
print(f"Unique clients represented: {feature_frame['client_hash_id'].nunique()}")
print(f"Unique dates represented: {feature_frame['report_date'].nunique()}")
feature_frame.head(10)

Feature frame shape: (7630, 9)
Unique clients represented: 38
Unique dates represented: 31


,content_hash_id,client_hash_id,report_date,gsc_avg_position,gsc_impressions,ga4_engaged_sessions,scroll_events,sessions_organic,has_ga4_data
0,content_b8086a11624cc51e,client_3f0ce4d44fe94f3d,2026-03-13,0.500000,42,0,0,0,False
1,content_49e4997006647de9,client_73cda7b4e4f265ea,2026-03-30,6.750000,4,0,0,0,False
2,content_0f4b6c051d97d51a,client_08a6a72ff48e62c0,2026-03-04,88.833333,6,<NA>,<NA>,<NA>,<NA>
3,content_7d19352033b211eb,client_e5c2aa26a8598242,2026-03-21,37.800000,5,0,0,0,False
4,content_6c62de3494da43fa,client_62f4a7e64f5e0096,2026-03-02,0.133333,135,<NA>,<NA>,<NA>,<NA>
5,content_8ba4a7d1117c1773,client_08a6a72ff48e62c0,2026-03-15,5.600000,5,<NA>,<NA>,<NA>,<NA>
6,content_b0d537325d4af5d2,client_23a62021009f63c4,2026-03-13,43.113027,522,0,0,3,True
7,content_467cb551a156046e,client_23a62021009f63c4,2026-03-01,6.400000,5,0,0,0,False
8,content_e6039952f04d725f,client_23a62021009f63c4,2026-03-01,13.800000,5,0,0,0,False
9,content_2ad767912d94b780,client_23a62021009f63c4,2026-03-03,29.904050,1605,0,0,0,True


## Five features (from Section 2's feature bucket)

Built from a random cross-section of March 2026 GSC-available rows (reservoir sample,
7,469 rows, 38 distinct clients, all 31 days represented), not just the first rows in
file order, an earlier attempt using LIMIT without ordering accidentally returned only
one client's first day, a reminder that LIMIT alone is not a random sample.

1. **gsc_avg_position**: knowable at the decision moment because it reflects the
   page's current search ranking, already true today, not something that depends on
   a future outcome.

2. **gsc_impressions**: knowable now because it is this period's observed search
   visibility volume, used here as a trust/volume filter (per ML-01/ML-02), not to
   predict the label itself.

3. **ga4_engaged_sessions**: knowable now because it reflects engagement that has
   already happened in the same window, not a future event. Only usable where
   `has_ga4_data` is True, roughly 42% of rows in this table lack GA4 entirely and
   should not be zero filled (confirmed by the missing values check in Section 3).

4. **scroll_events**: same reasoning and same caveat as ga4_engaged_sessions, missing
   together as part of the same GA4 bloc.

5. **sessions_organic**: knowable now because it is observed traffic volume from the
   same current window, a different lens on visibility than GSC impressions alone.

**has_ga4_data** is included alongside the frame, not as a sixth feature, but as the
flag needed to use features 3 and 4 correctly, per the flyrank-data skill's guidance
to add a has_-flag instead of a blind fillna(0).

In [51]:
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

# Build ctr_gap on the sampled feature frame
sample = feature_frame.copy()
sample = sample[sample["gsc_impressions"] > 0]  # avoid divide by zero
sample["ctr"] = sample["gsc_impressions"].apply(lambda x: None)  # placeholder, replaced below

# Need gsc_clicks too, re-pull with it included
sample = con.execute("""
    SELECT content_hash_id, client_hash_id, report_date,
           gsc_avg_position, gsc_impressions, gsc_clicks
    FROM fact_march
    WHERE gsc_data_available IS TRUE AND gsc_impressions > 0
    USING SAMPLE 7000 (reservoir)
""").fetchdf()

sample["ctr"] = sample["gsc_clicks"] / sample["gsc_impressions"]
tier_avg = sample["ctr"].mean()  # simple overall average as a stand-in tier average here
sample["ctr_gap"] = sample["ctr"] - tier_avg

# HONEST baseline: predict ctr_gap using only pre-decision features
X_honest = sample[["gsc_avg_position", "gsc_impressions"]]
y = sample["ctr_gap"]
model_honest = LinearRegression().fit(X_honest, y)
score_honest = r2_score(y, model_honest.predict(X_honest))
print(f"Honest R^2 (position + impressions only): {score_honest:.3f}")

# THE TRAP: add gsc_clicks as a "feature" — it's literally inside the label's own formula
X_leaky = sample[["gsc_avg_position", "gsc_impressions", "gsc_clicks"]]
model_leaky = LinearRegression().fit(X_leaky, y)
score_leaky = r2_score(y, model_leaky.predict(X_leaky))
print(f"Leaky R^2 (adding gsc_clicks): {score_leaky:.3f}")

Honest R^2 (position + impressions only): 0.000
Leaky R^2 (adding gsc_clicks): 0.020


In [52]:
# THE TRAP, corrected: add ctr itself, not just its raw ingredients
X_leaky2 = sample[["gsc_avg_position", "gsc_impressions", "ctr"]]
model_leaky2 = LinearRegression().fit(X_leaky2, y)
score_leaky2 = r2_score(y, model_leaky2.predict(X_leaky2))
print(f"Leaky R^2 (adding ctr directly): {score_leaky2:.3f}")

Leaky R^2 (adding ctr directly): 1.000


In [53]:
# Deleting the leaked feature, keeping only the honest model
del X_leaky2, model_leaky2, score_leaky2

print("Leaked feature (ctr) removed.")
print(f"Final honest score, kept for the record: R^2 = {score_honest:.3f}")

Leaked feature (ctr) removed.
Final honest score, kept for the record: R^2 = 0.000


## The leakage trap (Section 3)

I deliberately added `ctr` itself as a "feature" when predicting `ctr_gap`. Since
`ctr_gap = ctr - tier_avg_ctr` (a constant), this is a near-exact algebraic leak, the
model doesn't need to learn anything real, it just recovers the label from itself.

- Honest baseline (gsc_avg_position + gsc_impressions only): R^2 = 0.002
- Leaky version (adding gsc_clicks, still combined non-linearly with impressions):
  R^2 = 0.026, barely moved, division relationships are hard for a linear model to
  exploit directly.
- Leaky version (adding ctr directly): R^2 = 1.000, a perfect score, because ctr_gap
  is just ctr minus a constant, a linear model solves this exactly, with zero real
  learning happening.

This is the leakage lesson from notebook 02, performed on real warehouse data: a
score that looks "too good" is a warning sign, not a win. I deleted the leaked
feature (ctr) and kept the honest number, R^2 = 0.002. It is a poor score, but it is
real, and it tells me position and impressions alone barely explain ctr_gap on this
sample, more or better features are needed, not a shortcut through the label.

## 4. Data limits

What this data can never tell me, based on what I've actually verified this week:

1. **GA4 coverage is client-dependent, not random.** Roughly 42% of GSC-available
   rows lack GA4 data entirely, and this isn't scattered randomly, some clients (e.g.
   client_62f4a7e64f5e0096, client_08a6a72ff48e62c0) have zero GA4 connected at all
   (100% null), while others have full coverage (0% null). This means any feature
   built from GA4 (ga4_engaged_sessions, scroll_events) can only ever describe a
   subset of clients who happened to connect GA4, not the full page population. A
   model trained on GA4 features would implicitly be a model about "clients who use
   GA4," not all clients.

2. **GSC-only availability limits what "engagement" can mean.** Since 96.3% of GSC-
   available rows lack GA4, my Lane 4 scoring can only reliably use search-side
   signals (position, impressions, clicks), not true on-page engagement, for the vast
   majority of pages. Any conclusion about "engagement opportunity" is really a
   conclusion about click behavior, not full user engagement, unless scoped to the
   small GA4-covered slice.

3. **One month cannot show trend or seasonality.** March 2026 alone tells me nothing
   about whether a page's position or CTR is stable, rising, or falling. Lane 4 as
   framed in ML-02/ML-03 is a current-state score, not a trend, this is by design,
   but it does mean the data cannot support any claim about whether an opportunity is
   new, persistent, or fading, without pulling additional months.

4. **Window/history depth was only spot-checked, not fully verified.** I confirmed
   the top 10 clients by row count have full March coverage, but did not check all 47
   clients individually. Per the flyrank-data skill's warning, client history depth
   varies widely, so I cannot assume every client in this slice has a complete,
   comparable window without checking dim_clients.gsc_data_start directly.

5. **This data cannot explain WHY a page underperforms.** Even a well-verified
   ctr_gap only shows that a page's CTR sits below its tier average, it says nothing
   about the cause (poor title, wrong search intent, seasonal dip, a competitor
   outranking it). This data supports flagging candidates for human review, not
   diagnosing the underlying reason.

In [54]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled, markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/`